In [2]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")

dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }
    
dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }
    
# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath, drop_func) {
  df <- read.csv(filepath, header = TRUE)
  as.matrix(drop_func(df))
}
register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)


# Experiment on applying pseudobarcodes to THP1 macrophage

In [9]:
###################Whole Brain R1R2merged###################
df_DNA <- read.csv("read_counts_R1R2/THP1MacrophagePseudobarcodes_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1MacrophagePseudobarcodes_RNA_matched_barcodes_reshaped.csv", header=TRUE)

df_DNA <-as.matrix(dropEnhancer(df_DNA))
df_RNA <-as.matrix(dropEnhancer(df_RNA))

df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]

annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1MacrophagePseudobarcodes_barcodes.csv")

#Match row names
annot_DNA<-dropX(annot_DNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+ Test,
                          rnaDesign = ~ IFNB+IFNG+LPSIFNG+Naive_Macrophage+Allele_String,
                          reducedDesign = ~ IFNB+IFNG+LPSIFNG+Naive_Macrophage,
                          #mode="scale"
                          )
                          
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240417_comparative_THP1Pseudobarcodes_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-04-17 14:01:30.902592
Success: TRUE

Task duration:
   user  system elapsed 
 65.107   0.874  69.791 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6244164 333.5   10246857 547.3 10246857 547.3
Vcells 11625069  88.7   31091163 237.3 31091146 237.3

Log messages:
INFO [2024-04-17 14:00:21] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 8
Node: 17
Timestamp: 2024-04-17 14:01:54.17607
Success: TRUE

Task duration:
   user  system elapsed 
 87.895   0.925  93.534 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6244613 333.5   10246857 547.3 10246857 547.3
Vcells 11628515  88.8   31091163 237.3 31091146 237.3

Log messages:
INFO [2024-04-17 14:00:20] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 10
Node: 15
Timestamp: 2024-04-17 14:0

In [3]:
process_datasets <- function(dna_file, rna_file, annot_file, neg_control_file) {
  
  # Function to drop enhancer or any specific column and set row names
  drop_and_rename <- function(df, col_name) {
    if (col_name %in% names(df)) {
      if (any(is.na(df[[col_name]])) || any(duplicated(df[[col_name]]))) {
        stop(paste("Column", col_name, "contains NA or duplicate values."))
      }
      row.names(df) <- df[[col_name]]
      df <- df[, !(names(df) %in% c(col_name))]
    }
    df
  }

  # Load and process CSV files
  load_and_process <- function(filepath, drop_func, col_name) {
    df <- read.csv(filepath, header = TRUE)
    drop_func(df, col_name)
  }

  # Load data
  df_DNA <- load_and_process(dna_file, drop_and_rename, "enhancer_id")
  df_RNA <- load_and_process(rna_file, drop_and_rename, "enhancer_id")
  annot_DNA <- load_and_process(annot_file, drop_and_rename, "X")

  # Calculate selected columns for subsetting
  total_columns <- ncol(df_DNA)
  selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

  # Subset data starting from the 32nd row
  df_DNA <- df_DNA[32:nrow(df_DNA), selected_columns]
  df_RNA <- df_RNA[32:nrow(df_RNA), selected_columns]
  annot_DNA <- annot_DNA[selected_columns,]

  # Convert columns to factor for MPRAnalyze
  annot_DNA[] <- lapply(annot_DNA, as.factor)

  # Load negative controls and extract sequence IDs
  negative <- read.csv(neg_control_file, sep = "\t", header = FALSE)
  control <- as.character(negative$V1)  # V1 is the sequence_ID
  
  # Return a list of processed data frames and control
  list(DNA = as.matrix(df_DNA), RNA = as.matrix(df_RNA), Annotation = annot_DNA, Control = control)
}


In [11]:
results <- process_datasets(
    "read_counts_R1R2/THP1_Naive_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/THP1_Naive_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_THP1_Naive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_Naive_allele.csv")

##################################################################################################
results <- process_datasets(
    "read_counts_R1R2/THP1_LPSIFNG_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/THP1_LPSIFNG_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_THP1_LPSIFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )                  
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_LPSIFNG_allele.csv")

##################################################################################################
results <- process_datasets(
    "read_counts_R1R2/THP1_IFNB_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/THP1_IFNB_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNB_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
 #                 control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )                      
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_IFNB_allele.csv")

##################################################################################################
results <- process_datasets(
    "read_counts_R1R2/THP1_IFNG_DNA_matched_barcodes_reshaped.csv",
    "read_counts_R1R2/THP1_IFNG_RNA_matched_barcodes_reshaped.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_IFNG_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 18
Node: 7
Timestamp: 2024-08-13 13:59:04.196953
Success: TRUE

Task duration:
   user  system elapsed 
 59.356   0.336  59.707 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6303724 336.7   12115595 647.1  8420956 449.8
Vcells 11762993  89.8   22361811 170.7 18568176 141.7

Log messages:
INFO [2024-08-13 13:58:04] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 12
Node: 13
Timestamp: 2024-08-13 13:59:04.961107
Success: TRUE

Task duration:
   user  system elapsed 
 59.800   0.391  60.191 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6305070 336.8   12115595 647.1  8420956 449.8
Vcells 11768459  89.8   22361811 170.7 18568176 141.7

Log messages:
INFO [2024-08-13 13:58:04] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 16
Node: 9
Timestamp: 2024-08-13 13:

In [3]:
df_DNA <- read.csv("read_counts_R1R2/THP1_MacrophagevsMonocyte_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1_MacrophagevsMonocyte_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_MacrophagevsMonocyte_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+Naive_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          )                       
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_MacrophagevsMonocyte_interaction.csv")

#################################################################################################################
df_DNA <- read.csv("read_counts_R1R2/THP1_IFNBvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1_IFNBvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_IFNBvsNaive_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )    
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNB_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )
#alpha<-getAlpha(obj, by.factor = "Allele")                         
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_IFNBvsNaive_interaction.csv")

#################################################################################################################
df_DNA <- read.csv("read_counts_R1R2/THP1_IFNGvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1_IFNGvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_IFNGvsNaive_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                       
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_IFNGvsNaive_interaction.csv")

#################################################################################################################
df_DNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsIFNG_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsIFNG_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsIFNG_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                    
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_LPSIFNGvsIFNG_interaction.csv")

#################################################################################################################
df_DNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsNaive_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )

#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                      
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_LPSIFNGvsNaive_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-01-30 15:03:25.63967
Success: TRUE

Task duration:
   user  system elapsed 
117.148   0.881 122.839 

Memory used:
           used (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6289871  336   11968424 639.2  7293340 389.6
Vcells 13229317  101   27800372 212.2 23100304 176.3

Log messages:
INFO [2024-01-30 15:01:22] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 15
Node: 10
Timestamp: 2024-01-30 15:04:13.128808
Success: TRUE

Task duration:
   user  system elapsed 
166.624   0.803 170.642 

Memory used:
           used (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6290329  336   11968424 639.2  7293340 389.6
Vcells 13234713  101   27800372 212.2 23100304 176.3

Log messages:
INFO [2024-01-30 15:01:22] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 20
Timestamp: 2024-01-30 15:04:14.9

In [22]:
df_DNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsNaive_barcodes.csv")
df_DNA<-df_DNA[32:nrow(df_DNA), ]
df_RNA<-df_RNA[32:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )

#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                      
res <-testLrt(obj)
write.csv(res,"20240129_comparative_THP1_LPSIFNGvsNaive_interaction.csv")

Fitting model...



# Motif Only

In [2]:
process_datasets <- function(dna_file, rna_file, annot_file, neg_control_file) {
  
  # Function to drop enhancer or any specific column and set row names
  drop_and_rename <- function(df, col_name) {
    if (col_name %in% names(df)) {
      if (any(is.na(df[[col_name]])) || any(duplicated(df[[col_name]]))) {
        stop(paste("Column", col_name, "contains NA or duplicate values."))
      }
      row.names(df) <- df[[col_name]]
      df <- df[, !(names(df) %in% c(col_name))]
    }
    df
  }

  # Load and process CSV files
  load_and_process <- function(filepath, drop_func, col_name) {
    df <- read.csv(filepath, header = TRUE)
    drop_func(df, col_name)
  }

  # Load data
  df_DNA <- load_and_process(dna_file, drop_and_rename, "enhancer_id")
  df_RNA <- load_and_process(rna_file, drop_and_rename, "enhancer_id")
  annot_DNA <- load_and_process(annot_file, drop_and_rename, "X")

  # Calculate selected columns for subsetting
  total_columns <- ncol(df_DNA)
  selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

  # Subset data starting from the 32nd row
  df_DNA <- df_DNA[, selected_columns]
  df_RNA <- df_RNA[, selected_columns]
  annot_DNA <- annot_DNA[selected_columns,]

  # Convert columns to factor for MPRAnalyze
  annot_DNA[] <- lapply(annot_DNA, as.factor)

  # Load negative controls and extract sequence IDs
  negative <- read.csv(neg_control_file, sep = "\t", header = FALSE)
  control <- as.character(negative$V1)  # V1 is the sequence_ID
  
  # Return a list of processed data frames and control
  list(DNA = as.matrix(df_DNA), RNA = as.matrix(df_RNA), Annotation = annot_DNA, Control = control)
}


In [2]:
library("MPRAnalyze")
library("BiocParallel")

# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath) {
  df <- read.csv(filepath, header = TRUE,row.names = 1)
  as.matrix(df)
}

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/THP1Macrophage_DNA_matched_barcodes_reshaped_motif.csv")
df_RNA <- load_and_process("read_counts_R1R2/THP1Macrophage_RNA_matched_barcodes_reshaped_motif.csv")
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_THP1Macrophage_barcodes_motif.csv",row.names = 1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele+Tissue,
                          reducedDesign = ~ Tissue,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240611_comparative_THP1Macrophage_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-14 10:47:41.541908
Success: TRUE

Task duration:
    user   system  elapsed 
 159.116  531.596 1098.197 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5638721 301.2   10318407 551.1  9793577 523.1
Vcells 10915191  83.3   18137926 138.4 18134476 138.4

Log messages:
INFO [2024-06-14 10:29:24] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 19
Node: 6
Timestamp: 2024-06-14 11:06:46.735708
Success: TRUE

Task duration:
    user   system  elapsed 
 336.325 1187.821 2244.242 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5639705 301.2   10318407 551.1  9793577 523.1
Vcells 10918230  83.3   18137926 138.4 18134476 138.4

Log messages:
INFO [2024-06-14 10:29:23] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 13
Node: 12
Timestamp: 20

In [3]:
library("MPRAnalyze")
library("BiocParallel")

# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath) {
  df <- read.csv(filepath, header = TRUE,row.names = 1)
  as.matrix(df)
}

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/THP1_IFNG_DNA_matched_barcodes_motif.csv")
df_RNA <- load_and_process("read_counts_R1R2/THP1_IFNG_RNA_matched_barcodes_motif.csv")
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_THP1_IFNG_barcodes_motif.csv",row.names = 1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240922_comparative_THP1_IFNG_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-09-22 14:03:29.745682
Success: TRUE

Task duration:
   user  system elapsed 
 30.755 117.242 106.925 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6289871 336.0   12115625 647.1  7496627 400.4
Vcells 11386468  86.9   22361811 170.7 15787248 120.5

Log messages:
INFO [2024-09-22 14:01:43] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 21
Node: 4
Timestamp: 2024-09-22 14:05:52.692889
Success: TRUE

Task duration:
   user  system elapsed 
 84.718 260.873 250.151 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6290831 336.0   12115625 647.1  7496627 400.4
Vcells 11389312  86.9   22361811 170.7 15787248 120.5

Log messages:
INFO [2024-09-22 14:01:43] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 19
Node: 6
Timestamp: 2024-09-22 14:0

In [4]:
library("MPRAnalyze")
library("BiocParallel")

# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath) {
  df <- read.csv(filepath, header = TRUE,row.names = 1)
  as.matrix(df)
}

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/THP1_IFNB_DNA_matched_barcodes_motif.csv")
df_RNA <- load_and_process("read_counts_R1R2/THP1_IFNB_RNA_matched_barcodes_motif.csv")
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_THP1_IFNB_barcodes_motif.csv",row.names = 1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240922_comparative_THP1_IFNB_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-09-22 14:15:15.779821
Success: TRUE

Task duration:
   user  system elapsed 
 33.216  93.238  97.682 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6294824 336.2   12115625 647.1  7496627 400.4
Vcells 11398165  87.0   22361811 170.7 15787248 120.5

Log messages:
INFO [2024-09-22 14:13:38] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 13
Node: 12
Timestamp: 2024-09-22 14:16:47.801666
Success: TRUE

Task duration:
   user  system elapsed 
 75.305 194.172 191.081 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6294895 336.2   12115625 647.1  7496627 400.4
Vcells 11399039  87.0   22361811 170.7 15787248 120.5

Log messages:
INFO [2024-09-22 14:13:37] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 20
Timestamp: 2024-09-22 14:

In [5]:
library("MPRAnalyze")
library("BiocParallel")

# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath) {
  df <- read.csv(filepath, header = TRUE,row.names = 1)
  as.matrix(df)
}

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/THP1_LPSIFNG_DNA_matched_barcodes_motif.csv")
df_RNA <- load_and_process("read_counts_R1R2/THP1_LPSIFNG_RNA_matched_barcodes_motif.csv")
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNG_barcodes_motif.csv",row.names = 1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240922_comparative_THP1_LPSIFNG_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-09-22 14:26:28.742673
Success: TRUE

Task duration:
   user  system elapsed 
 37.614 106.682 125.922 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295132 336.2   12115625 647.1  7496627 400.4
Vcells 11398306  87.0   22361811 170.7 15787248 120.5

Log messages:
INFO [2024-09-22 14:24:23] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 11
Node: 14
Timestamp: 2024-09-22 14:27:58.6215
Success: TRUE

Task duration:
   user  system elapsed 
 82.420 219.536 217.265 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295203 336.3   12115625 647.1  7496627 400.4
Vcells 11399180  87.0   22361811 170.7 15787248 120.5

Log messages:
INFO [2024-09-22 14:24:21] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 21
Node: 4
Timestamp: 2024-09-22 14:28

In [9]:
library("MPRAnalyze")
library("BiocParallel")

# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath) {
  df <- read.csv(filepath, header = TRUE,row.names = 1)
  as.matrix(df)
}

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/THP1_Naive_DNA_matched_barcodes_motif.csv")
df_RNA <- load_and_process("read_counts_R1R2/THP1_Naive_RNA_matched_barcodes_motif.csv")
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_THP1_Naive_barcodes_motif.csv",row.names = 1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240922_comparative_THP1_Naive_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-09-22 14:41:20.179659
Success: TRUE

Task duration:
   user  system elapsed 
 36.618  88.121 100.679 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6306671 336.9   12115625 647.1  8939175 477.5
Vcells 11541529  88.1   22361811 170.7 19594830 149.5

Log messages:
INFO [2024-09-22 14:39:40] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 18
Node: 7
Timestamp: 2024-09-22 14:43:21.978907
Success: TRUE

Task duration:
   user  system elapsed 
 86.120 229.753 223.402 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6306742 336.9   12115625 647.1  8939175 477.5
Vcells 11542424  88.1   22361811 170.7 19594830 149.5

Log messages:
INFO [2024-09-22 14:39:39] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 20
Node: 5
Timestamp: 2024-09-22 14:4

# Allele only

In [1]:
process_datasets <- function(dna_file, rna_file, annot_file, neg_control_file) {
  # Function to drop enhancer or any specific column and set row names
  drop_and_rename <- function(df, col_name) {
    if (col_name %in% names(df)) {
      if (any(is.na(df[[col_name]])) || any(duplicated(df[[col_name]]))) {
        stop(paste("Column", col_name, "contains NA or duplicate values."))
      }
      row.names(df) <- df[[col_name]]
      df <- df[, !(names(df) %in% c(col_name))]
    }
    df
  }

  # Load and process CSV files
  load_and_process <- function(filepath, drop_func, col_name) {
    df <- read.csv(filepath, header = TRUE)
    drop_func(df, col_name)
  }

  # Load data
  df_DNA <- load_and_process(dna_file, drop_and_rename, "enhancer_id")
  df_RNA <- load_and_process(rna_file, drop_and_rename, "enhancer_id")
  annot_DNA <- load_and_process(annot_file, drop_and_rename, "X")

  # Calculate selected columns for subsetting
  total_columns <- ncol(df_DNA)
  selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]

  # Subset data starting from the 32nd row
  df_DNA <- df_DNA[, selected_columns]
  df_RNA <- df_RNA[, selected_columns]
  annot_DNA <- annot_DNA[selected_columns,]

  # Convert columns to factor for MPRAnalyze
  annot_DNA[] <- lapply(annot_DNA, as.factor)

  # Load negative controls and extract sequence IDs
  negative <- read.csv(neg_control_file, sep = "\t", header = FALSE)
  control <- as.character(negative$V1)  # V1 is the sequence_ID
  
  # Return a list of processed data frames and control
  list(DNA = as.matrix(df_DNA), RNA = as.matrix(df_RNA), Annotation = annot_DNA, Control = control)
}

In [5]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1Macrophage_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1Macrophage_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1Macrophage_barcodes.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String,
                          reducedDesign = ~ Tissue,
                          )                     
res <-testLrt(obj)
write.csv(res,"20240813_comparative_THP1Macrophage_alleleOnly.csv")


Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 11:39:58.11587
Success: TRUE

Task duration:
   user  system elapsed 
259.173   2.033 290.381 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6175702 329.9   11606934 619.9  8605685 459.6
Vcells 13757620 105.0   27813248 212.2 27802250 212.2

Log messages:
INFO [2024-08-13 11:35:07] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 15
Node: 10
Timestamp: 2024-08-13 11:41:11.643937
Success: TRUE

Task duration:
   user  system elapsed 
335.611   1.858 364.229 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6176981 329.9   11606934 619.9  8605685 459.6
Vcells 13764439 105.1   27813248 212.2 27802250 212.2

Log messages:
INFO [2024-08-13 11:35:07] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 14
Node: 11
Timestamp: 2024-08-13 11:

In [10]:


results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_Naive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_Naive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_Naive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_Naive_alleleOnly.csv")

results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNB_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNB_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNB_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_IFNB_alleleOnly.csv")

results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_LPSIFNG_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_LPSIFNG_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_LPSIFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_LPSIFNG_alleleOnly.csv")

##############################################################
results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNG_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNG_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNG_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_THP1_IFNG_alleleOnly.csv")



results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNBvsNaive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNBvsNaive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNBvsNaive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNB_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                 
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_IFNBvsNaive_interaction.csv")


results <- process_datasets(
    "read_counts_R1R2/allele_only/THP1_IFNGvsNaive_DNA_matched_barcodes_reshaped_allele.csv",
    "read_counts_R1R2/allele_only/THP1_IFNGvsNaive_RNA_matched_barcodes_reshaped_allele.csv",
    "annotation_barcodes/mpra3_annot_THP1_IFNGvsNaive_barcodes.csv",
    "indexing/mpra3_negatives.csv")
df_DNA <- results[[1]]
df_RNA <- results[[2]]
annot_DNA <-results[[3]]
control <-results[[4]]

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+IFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )               
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_IFNGvsNaive_interaction.csv")


df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsIFNG_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsIFNG_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsIFNG_barcodes.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                     
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_LPSIFNGvsIFNG_interaction.csv")

df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsNaive_barcodes.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
##############################################################

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   


obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+LPSIFNG_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )              
res <-testLrt(obj)
write.csv(res,"20240812_comparative_THP1_LPSIFNGvsNaive_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-17 17:42:33.342637
Success: TRUE

Task duration:
    user   system  elapsed 
1019.238 3692.237 9238.075 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5710399 305.0    9001995 480.8  9001995 480.8
Vcells 11410494  87.1   22045152 168.2 22044750 168.2

Log messages:
INFO [2024-06-17 15:08:36] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 16
Node: 9
Timestamp: 2024-06-17 18:20:53.83615
Success: TRUE

Task duration:
     user    system   elapsed 
 1285.305  4691.823 11541.216 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5711485 305.1    9001995 480.8  9001995 480.8
Vcells 11415689  87.1   22045152 168.2 22044750 168.2

Log messages:
INFO [2024-06-17 15:08:33] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 23
Node: 2
Timestamp